In [ ]:
!pip install --no-build-isolation git+https://github.com/OpenAccess-AI-Collective/axolotl.git


In [4]:
!pip install axolotl[flash-attn]


In [ ]:
!pip install "cut-cross-entropy[transformers] @ git+https://github.com/axolotl-ai-cloud/ml-cross-entropy.git@318b7e2"


In [13]:
dataset_id = 'winglian/pirate-ultrachat-10k'

In [14]:
dataset_id

'winglian/pirate-ultrachat-10k'

In [ ]:
dataset_id['train'][0]

In [8]:
import os
os.environ['AXOLOTL_DO_NOT_TRACK'] = '1'

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [9]:
from axolotl.utils import set_pytorch_cuda_alloc_conf

set_pytorch_cuda_alloc_conf()

In [10]:
from axolotl.cli.config import load_cfg

In [11]:
from axolotl.utils.dict import DictDefault

In [23]:

config = DictDefault(
    base_model="Qwen/Qwen2.5-3B-Instruct",
    load_in_4bit=True,
    adapter="qlora",
    lora_r=32,
    lora_alpha=64,
    lora_target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "down_proj",
        "up_proj",
    ],
    lora_qkv_kernel=False,
    lora_o_kernel=False,
    lora_mlp_kernel=False,
    embeddings_skip_upcast=True,
    xformers_attention=True,
    plugins=[],
    sample_packing=False,
    learning_rate=0.00019,
    sequence_len=1024,
    micro_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },
    optimizer="paged_adamw_8bit",
    lr_scheduler="cosine",
    warmup_steps=5,
    fp16=True,
    bf16=False,
    max_grad_norm=0.1,
    num_epochs=1,
    saves_per_epoch=2,
    logging_steps=1,
    output_dir="./outputs/qwen-sft-pirate-rrr",
    chat_template="qwen3",
    datasets=[
        {
            "path": dataset_id,
            "type": "chat_template",
            "split": "train",
            "eot_tokens": ["<|im_end|>"],
        }
    ],
    dataloader_prefetch_factor=2,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

In [25]:
cgf= load_cfg(config=config)

In [26]:
from axolotl.common.datasets import load_datasets

dataset_metadata = load_datasets(cfg = cgf)

In [21]:
dataset_metadata

TrainDatasetMeta(train_dataset=Dataset({
    features: ['input_ids', 'labels', 'attention_mask'],
    num_rows: 8840
}), eval_dataset=None, total_num_steps=1105)

In [27]:
from axolotl.train import train
model, tokenizer, trainer = train(cfg =  cgf, dataset_meta = dataset_metadata)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

AssertionError: Torch not compiled with CUDA enabled

In [28]:
from transformers import TextStreamer
messages = [
    {
        "role": "user",
        "content": "Explain the Pythagorean theorem to me.",
    },
]
prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
    enable_thinking=False,
)

outputs = model.generate(
    **tokenizer(prompt, return_tensors="pt").to("cuda"),
    max_new_tokens=192,
    temperature=1.0,
    top_p=0.8,
    top_k=32,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)



NameError: name 'tokenizer' is not defined

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!hf auth upload --repo-type=model winglian/pirate-qwen-14B "./outputs/qwen-sft-pirate-rrr"
